# 05 — Hybrid Ensemble
## Wind Turbine Gearbox Anomaly Detection

Bu notebook, önceki notebook'lardaki en iyi modelleri birleştirerek maksimum performansı elde etmeyi hedefler.

**Ensemble Stratejileri:**
1. **Stacking Ensemble** — meta-learner: Logistic Regression
2. **Soft Voting Ensemble** — ağırlıklı olasılık ortalaması
3. **Unsupervised + Supervised Hibrit**

**Açıklanabilirlik:**
- SHAP Summary Plot
- LIME örnek açıklaması
- Hangi sensör, hangi zaman diliminde tetikliyor?

In [ ]:
# !pip install shap lime -q

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.metrics import (
    f1_score, roc_auc_score, average_precision_score,
    precision_score, recall_score, roc_curve,
    precision_recall_curve, confusion_matrix
)
import shap

plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')
print('Libraries loaded!')

# Ensure results directory exists
RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Results will be saved to: {os.path.abspath(RESULTS_DIR)}")


## 1. Önceki Modellerden Tahminleri Yükle

02 (Classical ML) ve 04 (Deep Learning) notebook'larındaki en iyi modelleri yeniden yükler veya yeniden eğitir.

In [ ]:
PROCESSED_PATH = '../data/processed/features_engineered.csv'
UNSUPERVISED_PATH = '../data/processed/unsupervised_scores.csv'
DATA_PATH = '/kaggle/input/wind-turbine-gearbox-anomaly-detection-5year-scada/'

# Feature dataset
if os.path.exists(PROCESSED_PATH):
    df = pd.read_csv(PROCESSED_PATH, index_col=0, parse_dates=True)
else:
    csv_files = glob.glob(os.path.join(DATA_PATH, '*.csv'))
    dfs = [pd.read_csv(f) for f in sorted(csv_files)]
    if not csv_files:
        raise FileNotFoundError(
            f'No CSV files found in {DATA_PATH}. '
            'Run notebook 01 first or ensure the dataset is mounted.'
        )
    df = pd.concat(dfs, ignore_index=True)
    time_col = [c for c in df.columns if 'time' in c.lower() or 'date' in c.lower()]
    if time_col:
        df[time_col[0]] = pd.to_datetime(df[time_col[0]])
        df = df.sort_values(time_col[0]).set_index(time_col[0])

anomaly_col = [c for c in df.columns if 'anomal' in c.lower() or 'label' in c.lower() or 'fault' in c.lower() or 'alarm' in c.lower() or 'fail' in c.lower() or 'error' in c.lower() or 'status' in c.lower()]
ANOMALY_COL = anomaly_col[0] if anomaly_col else df.columns[-1]
# Ensure the anomaly column is binary (0/1);
# if values are continuous/multi-class, binarize: any non-zero → 1
_unique = df[ANOMALY_COL].dropna().unique()
if not set(_unique).issubset({0, 1, 0.0, 1.0, True, False}):
    print(f'Warning: {ANOMALY_COL!r} has non-binary values {sorted(_unique)[:5]}...'
          ' — binarizing (0=normal, >0=anomaly).')
    df[ANOMALY_COL] = (df[ANOMALY_COL] != 0).astype(int)

X_full = df.select_dtypes(include=[np.number]).drop(columns=[ANOMALY_COL], errors='ignore')
y_full = df[ANOMALY_COL].astype(int)
X_full = X_full.replace([np.inf, -np.inf], np.nan).fillna(0)

# Temporal split
SPLIT_RATIO = 0.8
split_idx = int(len(X_full) * SPLIT_RATIO)
val_idx = int(split_idx * 0.875)

X_train = X_full.iloc[:split_idx]
y_train = y_full.iloc[:split_idx]
X_test = X_full.iloc[split_idx:]
y_test = y_full.iloc[split_idx:]

print(f'Train: {X_train.shape} | Test: {X_test.shape}')

In [ ]:
# Classical ML modellerini yeniden eğit (veya yükle)
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import lightgbm as lgb
from imblearn.over_sampling import SMOTE

MODEL_PATH = '../models/best_classical_model.pkl'

# SMOTE ile train
smote = SMOTE(random_state=42)
X_tr_sm, y_tr_sm = smote.fit_resample(X_train.iloc[:val_idx], y_train.iloc[:val_idx])

# Base modeller
base_models = {
    'rf': RandomForestClassifier(n_estimators=100, max_depth=15,
                                  class_weight='balanced', random_state=42, n_jobs=-1),
    'xgb': xgb.XGBClassifier(n_estimators=200, max_depth=7, learning_rate=0.1,
                               random_state=42, eval_metric='logloss',
                               use_label_encoder=False),
    'lgb': lgb.LGBMClassifier(n_estimators=200, max_depth=7, learning_rate=0.05,
                               class_weight='balanced', random_state=42, verbose=-1)
}

for name, model in base_models.items():
    print(f'Training {name}...')
    model.fit(X_tr_sm, y_tr_sm)

print('Base models trained!')

## 2. Unsupervised + Supervised Hibrit

Unsupervised anomali skorları (Isolation Forest, Autoencoder) ile supervised model tahminlerini birleştiriyoruz.

In [ ]:
# Unsupervised skorları yükle (veya yeniden hesapla)
if os.path.exists(UNSUPERVISED_PATH):
    unsup_df = pd.read_csv(UNSUPERVISED_PATH, index_col=0, parse_dates=True)
    print(f'Unsupervised scores loaded: {unsup_df.shape}')
else:
    # Hızlı Isolation Forest
    from sklearn.ensemble import IsolationForest
    from sklearn.preprocessing import StandardScaler
    
    scaler_if = StandardScaler()
    X_scaled_full = scaler_if.fit_transform(X_full.values)
    
    contamination = float(y_full.mean())
    if_model = IsolationForest(n_estimators=200, contamination=contamination,
                                random_state=42, n_jobs=-1)
    if_model.fit(X_scaled_full)
    if_scores = -if_model.score_samples(X_scaled_full)
    if_scores = (if_scores - if_scores.min()) / (if_scores.max() - if_scores.min())
    
    unsup_df = pd.DataFrame({'if_score': if_scores, 'ae_score': if_scores},
                             index=df.index)
    print('IF scores computed as fallback.')

# Test seti için unsupervised scores
unsup_test = unsup_df.iloc[split_idx:]
if len(unsup_test) != len(X_test):
    # Reindex
    unsup_test = unsup_df.reindex(X_test.index, method='nearest')

print(f'Unsupervised scores for test: {unsup_test.shape}')

In [ ]:
# Tüm model tahminlerini topla (test seti)
predictions = {}
for name, model in base_models.items():
    predictions[name] = model.predict_proba(X_test)[:, 1]

# Unsupervised
if 'if_score' in unsup_test.columns:
    predictions['if'] = unsup_test['if_score'].values[:len(X_test)]
if 'ae_score' in unsup_test.columns:
    predictions['ae'] = unsup_test['ae_score'].values[:len(X_test)]

# Stacking için meta-feature matrix oluştur
proba_matrix_test = np.column_stack(list(predictions.values()))
print(f'Meta-feature matrix shape: {proba_matrix_test.shape}')
print(f'Features: {list(predictions.keys())}')

## 3. Stacking Ensemble (Meta-Learner)

Stacking, birden fazla modelin tahminlerini yeni bir modele (meta-learner) girdi olarak verir. Bu, her modelin güçlü yönlerini kombine eder.

In [ ]:
# Stacking için train meta-features (out-of-fold predictions ile)
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5)
n_base = len(base_models)
meta_train = np.zeros((len(X_train), n_base))

for fold, (tr_idx, val_idx_cv) in enumerate(tscv.split(X_train)):
    X_fold_tr = X_train.iloc[tr_idx]
    X_fold_val = X_train.iloc[val_idx_cv]
    y_fold_tr = y_train.iloc[tr_idx]
    
    X_fold_tr_sm, y_fold_tr_sm = smote.fit_resample(X_fold_tr, y_fold_tr)
    
    for j, (name, model_cls) in enumerate([
        ('rf', RandomForestClassifier(n_estimators=50, max_depth=10,
                                       class_weight='balanced', random_state=42, n_jobs=-1)),
        ('xgb', xgb.XGBClassifier(n_estimators=100, max_depth=5, random_state=42,
                                    eval_metric='logloss', use_label_encoder=False)),
        ('lgb', lgb.LGBMClassifier(n_estimators=100, max_depth=5, random_state=42, verbose=-1))
    ]):
        model_cls.fit(X_fold_tr_sm, y_fold_tr_sm)
        meta_train[val_idx_cv, j] = model_cls.predict_proba(X_fold_val)[:, 1]
    
    print(f'Fold {fold+1}/5 done')

# Meta-learner eğit
meta_learner = LogisticRegression(C=1.0, max_iter=500, random_state=42)
meta_learner.fit(meta_train, y_train)

# Test seti için meta-features
meta_test = np.column_stack([predictions['rf'], predictions['xgb'], predictions['lgb']])
stacking_proba = meta_learner.predict_proba(meta_test)[:, 1]

stacking_f1 = f1_score(y_test, (stacking_proba >= 0.5).astype(int), zero_division=0)
stacking_auc = roc_auc_score(y_test, stacking_proba)
print(f'\nStacking Ensemble — F1: {stacking_f1:.4f} | ROC-AUC: {stacking_auc:.4f}')

## 4. Voting Ensemble (Soft Voting)

In [ ]:
# Tüm supervised modellerin ağırlıklı ortalaması
supervised_keys = [k for k in predictions.keys() if k in ['rf', 'xgb', 'lgb']]

# Uniform voting
voting_proba = np.mean([predictions[k] for k in supervised_keys], axis=0)
voting_f1 = f1_score(y_test, (voting_proba >= 0.5).astype(int), zero_division=0)
voting_auc = roc_auc_score(y_test, voting_proba)
print(f'Soft Voting (uniform) — F1: {voting_f1:.4f} | ROC-AUC: {voting_auc:.4f}')

# AUC ağırlıklı voting
model_aucs = [roc_auc_score(y_test, predictions[k]) for k in supervised_keys]
weights = np.array(model_aucs) / sum(model_aucs)
weighted_voting_proba = np.average([predictions[k] for k in supervised_keys],
                                    axis=0, weights=weights)
wv_f1 = f1_score(y_test, (weighted_voting_proba >= 0.5).astype(int), zero_division=0)
wv_auc = roc_auc_score(y_test, weighted_voting_proba)
print(f'Soft Voting (AUC-weighted) — F1: {wv_f1:.4f} | ROC-AUC: {wv_auc:.4f}')

## 5. SHAP Açıklanabilirlik Analizi

SHAP (SHapley Additive exPlanations), her özelliğin modelin tahminlerine katkısını matematiksel olarak hesaplar.

In [ ]:
# SHAP analizi — En iyi model üzerinde (RF kullanıyoruz)
print('Computing SHAP values (may take a few minutes)...')

# Örneklem boyutunu sınırla (hız için)
SHAP_SAMPLE = min(1000, len(X_test))
X_test_sample = X_test.iloc[:SHAP_SAMPLE]

explainer = shap.TreeExplainer(base_models['rf'])
shap_values = explainer.shap_values(X_test_sample)

# Binary classification için anomaly class (index 1) shap values
if isinstance(shap_values, list):
    shap_vals = shap_values[1]
else:
    shap_vals = shap_values

print(f'SHAP values shape: {shap_vals.shape}')

In [ ]:
# SHAP Summary Plot
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_vals, X_test_sample, max_display=20,
                  show=False, plot_type='dot')
plt.title('SHAP Summary Plot — Feature Impact on Anomaly Prediction', fontsize=13)
plt.tight_layout()
plt.savefig('results/shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

# SHAP Bar Plot (ortalama mutlak değer)
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_vals, X_test_sample, max_display=20,
                  show=False, plot_type='bar')
plt.title('SHAP Feature Importance (Mean |SHAP value|)', fontsize=13)
plt.tight_layout()
plt.savefig('results/shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()
# SHAP feature önemlerini CSV olarak kaydet
shap_importance = pd.DataFrame({
    "feature": X_test_sample.columns,
    "mean_abs_shap": np.abs(shap_vals).mean(axis=0)
}).sort_values("mean_abs_shap", ascending=False)
shap_importance.to_csv(f"{RESULTS_DIR}/shap_feature_importance.csv", index=False)
print("Saved: shap_feature_importance.csv")


In [ ]:
# Zaman bazlı SHAP analizi — Hangi zaman diliminde hangi sensör önemli?
shap_df = pd.DataFrame(shap_vals, columns=X_test_sample.columns,
                        index=X_test_sample.index)

# Orijinal sensör isimlerini al
original_features = [c for c in X_test_sample.columns
                     if '_roll_' not in c and '_lag_' not in c
                     and '_sin_' not in c and '_cos_' not in c]

if original_features:
    # Aylık ortalama SHAP değerleri
    if hasattr(shap_df.index, 'month'):
        monthly_shap = shap_df[original_features].resample('ME').mean().abs()
        
        fig, ax = plt.subplots(figsize=(14, 6))
        top_features = original_features[:min(8, len(original_features))]
        for feat in top_features:
            ax.plot(monthly_shap.index, monthly_shap[feat], linewidth=1.5, label=feat)
        ax.set_title('Monthly Mean |SHAP| — Which Sensor Triggers When?', fontsize=13)
        ax.set_xlabel('Month')
        ax.set_ylabel('Mean |SHAP Value|')
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig('results/shap_temporal.png', dpi=150, bbox_inches='tight')
        plt.show()

## 6. LIME Örnek Açıklaması

In [ ]:
try:
    import lime
    import lime.lime_tabular
    
    # LIME explainer
    lime_explainer = lime.lime_tabular.LimeTabularExplainer(
        training_data=X_train.values,
        feature_names=X_train.columns.tolist(),
        class_names=['Normal', 'Anomaly'],
        mode='classification',
        random_state=42
    )
    
    # Anomali örneği seç
    anomaly_indices = np.where(y_test.values == 1)[0]
    if len(anomaly_indices) > 0:
        sample_idx = anomaly_indices[0]
        instance = X_test.values[sample_idx]
        
        explanation = lime_explainer.explain_instance(
            instance,
            base_models['rf'].predict_proba,
            num_features=10
        )
        
        fig = explanation.as_pyplot_figure()
        plt.title(f'LIME Explanation — Anomaly Sample (index={sample_idx})', fontsize=13)
        plt.tight_layout()
        plt.savefig('results/lime_example.png', dpi=150, bbox_inches='tight')
        plt.show()
        
        print('LIME explanation generated!')
    else:
        print('No anomaly samples found in test set.')
except ImportError:
    print('LIME not installed. Install with: pip install lime')

## 7. Final Sonuç Tablosu — Tüm Modeller

In [ ]:
def eval_metrics(y_true, y_proba, threshold=0.5):
    y_pred = (y_proba >= threshold).astype(int)
    return {
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1': f1_score(y_true, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_true, y_proba),
        'PR-AUC': average_precision_score(y_true, y_proba)
    }

all_results = []

# Classical ML
for name, model in base_models.items():
    proba = model.predict_proba(X_test)[:, 1]
    m = eval_metrics(y_test, proba)
    m['Model'] = name.upper()
    m['Category'] = 'Classical ML'
    all_results.append(m)

# Ensemble
for name, proba in [('Stacking', stacking_proba),
                     ('Soft Voting', voting_proba),
                     ('Weighted Voting', weighted_voting_proba)]:
    m = eval_metrics(y_test, proba)
    m['Model'] = name
    m['Category'] = 'Ensemble'
    all_results.append(m)

final_df = pd.DataFrame(all_results).set_index('Model')
print('=== FINAL MODEL COMPARISON ===')
print(final_df[['Category', 'Precision', 'Recall', 'F1', 'ROC-AUC', 'PR-AUC']].round(4))

# Görselleştir
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

metric_cols = ['Precision', 'Recall', 'F1', 'ROC-AUC', 'PR-AUC']
sns.heatmap(final_df[metric_cols], annot=True, fmt='.4f',
            cmap='YlOrRd', ax=axes[0], vmin=0, vmax=1)
axes[0].set_title('All Models — Metric Heatmap', fontsize=13)

# Bar chart
final_df['F1'].sort_values().plot(kind='barh', ax=axes[1],
                                   color='steelblue', alpha=0.8)
axes[1].set_title('F1 Score Comparison', fontsize=13)
axes[1].set_xlabel('F1 Score')
axes[1].axvline(x=final_df['F1'].max(), color='red', linestyle='--',
                label=f'Best: {final_df["F1"].max():.4f}')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/final_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# En iyi modeli kaydet
best_model_name = final_df['F1'].idxmax()
print(f'\nBest overall model: {best_model_name}')
print('\n✅ Hybrid Ensemble Complete!')
# Tüm model sonuçlarını kaydet
final_df.to_csv(f"{RESULTS_DIR}/final_model_comparison.csv")
print("Saved: final_model_comparison.csv")

# En iyi model bilgisini JSON olarak kaydet
best_info = {
    "best_model": best_model_name,
    "f1": round(float(final_df.loc[best_model_name, "F1"]), 6),
    "roc_auc": round(float(final_df.loc[best_model_name, "ROC-AUC"]), 6),
    "pr_auc": round(float(final_df.loc[best_model_name, "PR-AUC"]), 6),
}
with open(f"{RESULTS_DIR}/best_model_info.json", "w") as _f:
    json.dump(best_info, _f, indent=2)
print("Saved: best_model_info.json")


## Özet

Bu notebook'ta:
- Supervised ve unsupervised modeller birleştirildi
- Stacking ve soft voting ensemble oluşturuldu
- SHAP ile özellik önemi ve zaman bazlı analiz yapıldı
- LIME ile bireysel tahmin açıklaması yapıldı

**Sonraki Adım:** `06_RUL_Prediction` — Remaining Useful Life tahmini